# Hads Eye

HADS anxiety/depression scores vs eye-tracking measures.

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

columns_mapping = {
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
}

baseline_eye_tracking = pd.read_csv(f'{DATA}/sed.csv').rename(columns=columns_mapping)
eye_tracking_01 = pd.read_csv(f'{DATA}/sed_01.csv').rename(columns=columns_mapping)
eye_tracking_02 = pd.read_csv(f'{DATA}/sed_02.csv').rename(columns=columns_mapping)
eye_tracking_03 = pd.read_csv(f'{DATA}/sed_03.csv').rename(columns=columns_mapping)

psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

def clean_eye_tracking_data(eye_tracking_data):
    df = eye_tracking_data.copy()
    df['timestamp'] = pd.to_datetime(
        df['timestamp'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    return df.dropna(subset=['timestamp'])

baseline_eye_tracking = clean_eye_tracking_data(baseline_eye_tracking)
eye_tracking_01 = clean_eye_tracking_data(eye_tracking_01)
eye_tracking_02 = clean_eye_tracking_data(eye_tracking_02)
eye_tracking_03 = clean_eye_tracking_data(eye_tracking_03)

psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

def filter_eye_tracking_data(eye_tracking_data, question):
    return eye_tracking_data[
        (eye_tracking_data['timestamp'] >= question['Question Start Time']) &
        (eye_tracking_data['timestamp'] <= question['Question Answer Time'])
    ]

BLINK_THRESHOLD = 1.0
MIN_CLOSED_FRAMES = 3  # ~100ms at 30Hz

def count_blinks(series, threshold, min_frames):
    # sustained closure onset
    closed = (series <= threshold).astype(int)
    sustained = closed.rolling(min_frames).sum() == min_frames
    return int((sustained & ~sustained.shift(1, fill_value=False)).sum())

def calculate_eye_tracking_metrics(eye_tracking_data):
    average_pupil_dilation = eye_tracking_data['pupil_dilation'].mean()
    total_duration_minutes = (
        (eye_tracking_data['timestamp'].max() - eye_tracking_data['timestamp'].min())
        .total_seconds() / 60
    )
    if total_duration_minutes <= 0:
        return average_pupil_dilation, 0.0, 0.0
    left_blink_rate = count_blinks(eye_tracking_data['left_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes
    right_blink_rate = count_blinks(eye_tracking_data['right_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes
    return average_pupil_dilation, left_blink_rate, right_blink_rate

baseline_metrics = calculate_eye_tracking_metrics(baseline_eye_tracking)

def detect_significant_increase(test_metrics, baseline_metrics):
    return (
        test_metrics[0] > baseline_metrics[0],
        test_metrics[1] > baseline_metrics[1],
        test_metrics[2] > baseline_metrics[2]
    )

def filter_correct_questions(df, types_count):
    filtered_df = pd.DataFrame()
    for q_type, count in types_count.items():
        filtered_df = pd.concat([filtered_df, df[df['Type'] == q_type].head(count)])
    return filtered_df


In [ ]:
# Validate – correcting total number of questions per type
types_count = {
    'HADS': 14,
    'STAI-S': 20,
    'STAI-T': 20,
    'BFI': 10,
    'FQ': 24
}

questions_01 = filter_correct_questions(psychometric_01, types_count)
questions_02 = filter_correct_questions(psychometric_02, types_count)
questions_03 = filter_correct_questions(psychometric_03, types_count)

# Calculate metrics with baseline comparison (rounded, with per-metric flags)
def calculate_metrics_validated(questions, eye_tracking_data, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        filtered_data = filter_eye_tracking_data(eye_tracking_data, question)
        if not filtered_data.empty:
            metrics = calculate_eye_tracking_metrics(filtered_data)
            significant_increases = detect_significant_increase(metrics, baseline_metrics)
            results.append({
                'Type': question['Type'],
                'Question': question['Question'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'Average Pupil Dilation': round(metrics[0], 2),
                'Average Left Blink Rate': round(metrics[1], 2),
                'Average Right Blink Rate': round(metrics[2], 2),
                'Sign of Anxiety': 'Yes' if any(significant_increases) else 'No',
                'Pupil Dilation Increase': 'Yes' if significant_increases[0] else 'No',
                'Left Blink Rate Increase': 'Yes' if significant_increases[1] else 'No',
                'Right Blink Rate Increase': 'Yes' if significant_increases[2] else 'No'
            })
    return pd.DataFrame(results)

results_01 = calculate_metrics_validated(questions_01, eye_tracking_01, baseline_metrics)
results_02 = calculate_metrics_validated(questions_02, eye_tracking_02, baseline_metrics)
results_03 = calculate_metrics_validated(questions_03, eye_tracking_03, baseline_metrics)

# Combine and validate row count
results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)

expected_rows = sum(types_count.values()) * 3
if len(combined_results) != expected_rows:
    raise ValueError(f"Expected {expected_rows} rows, got {len(combined_results)}")

# Save data
combined_results.to_csv(f'{DATA}/QQ2.csv', index=False)

combined_results.head()